# 08 Feature Vs Raw

This notebook compares matrix profile behavior on a raw price-like series and on simple derived features. The goal is to build intuition for when level, local change, or rolling volatility gives a cleaner motif signal.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if candidate.name == "matrix_profile_lab":
        LAB_ROOT = candidate
        break
else:
    raise RuntimeError("Open this notebook from inside matrix_profile_lab.")

if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from utils.data_generators import repeated_pattern_series, rolling_feature_view
from utils.mp_helpers import compute_matrix_profile
from utils.plotting import plot_feature_grid


In [ ]:
base_series, motif, _ = repeated_pattern_series(
    length=260,
    motif_length=24,
    motif_positions=(50, 145, 215),
    noise=0.08,
    seed=27,
)

price_like = 100 + np.cumsum(0.15 * base_series + 0.03)
features = rolling_feature_view(price_like, window=10)

plot_feature_grid(
    features,
    columns=["price", "returns", "rolling_volatility"],
    title="Raw view versus simple feature views",
)

window = 24
raw_mp = compute_matrix_profile(features["price"].to_numpy(), window=window)
returns_mp = compute_matrix_profile(features["returns"].to_numpy(), window=window)
vol_mp = compute_matrix_profile(features["rolling_volatility"].to_numpy(), window=window)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(raw_mp["profile"], linewidth=2.0)
axes[0].set_title("Matrix profile on raw price-like values")
axes[1].plot(returns_mp["profile"], linewidth=2.0)
axes[1].set_title("Matrix profile on returns")
axes[2].plot(vol_mp["profile"], linewidth=2.0)
axes[2].set_title("Matrix profile on rolling volatility")

for ax in axes:
    ax.grid(alpha=0.25)
    ax.set_ylabel("profile")

axes[-1].set_xlabel("subsequence start")
fig.tight_layout()
